# 💼 The Analyst's Notebook · Part 11
### Trees and forests for the forecast

After Part 9 the risk report holds two results. The first is the volatility forecast it was built around: next month's volatility from this month's alone, the one-column model chosen in Part 5, which in Part 6 held off every column the desk could offer, 0.00412 against 0.00460 for ridge on twenty. The second is Part 9's jump warning: a one-column classifier with an AUC of 0.889, run at a threshold of 0.25 chosen from the desk's costs.

Every model in the report so far is linear in its columns: one slope per column, the same in a calm month as in a crash. This part tries a different kind of model. A tree asks a question of one column and forecasts differently on each side of the answer, and a forest averages many trees. Part 11 gives them the question the report was built on, next month's volatility, and holds them to what the report has already established: the one-column forecast, ridge, the folds and the desk. The last question takes the forest to Part 9's jump label, once.

**Course site:** [the lecture](https://theill95.github.io/mlfin-2026/session_11/session_11.html) · [the exercises, in Colab](https://colab.research.google.com/github/theill95/mlfin-2026/blob/main/session_11/session_11_exercises.ipynb) · [the cheatsheet](https://theill95.github.io/mlfin-2026/cheatsheet.html) · [all sessions](https://theill95.github.io/mlfin-2026/)

## How to work through this

- Run the **quick load** cell first. It brings back what the risk report holds after Part 9 and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.
- Q10 runs a grid search over forests and takes about a minute on Colab.

**A note on units.** The lecture worked in percent, on the index table. This notebook keeps the plain decimals of Parts 1 to 9 and stays on Apple, so every number compares directly with the report's. A tree does not mind the units: a cut at 0.0175 in decimals is the same cut as 1.75 in percent, and no column needs scaling.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The packages, the price table, and what the risk report holds after Part 9. Run it and read what it prints.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


def rmse(actual, predicted):
    """Root mean squared error, as in Part 6, as a plain number."""
    return float(np.sqrt(mean_squared_error(actual, predicted)))


# The whole universe: eleven instruments, 2015 to 2024, returns in plain decimals
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change()
TICKERS = sorted(prices["ticker"].unique())

folds = TimeSeriesSplit(n_splits=5)

# --- What the risk report holds after Part 9, on Apple ---
part9_split = "by date: train to 2022-12-31, test from 2023-01-01"

# the forecast: Part 5's target, Part 6's models
part5_target = "sd of daily returns over the next 20 trading days"
part6_one_rmse = 0.00412        # linear regression on vol_20d, the test block
part6_one_folds = 0.00755       # the same model on the five time-ordered folds
part6_ridge_rmse = 0.00460      # ridge on all twenty columns, alpha chosen on the folds
part6_ridge_folds = 0.00761     # the same search, its score on the folds
part6_desk = {                  # (ridge, one column) on each instrument's test block
    'AAPL': (0.00460, 0.00412),
    'DIS': (0.00559, 0.00645),
    'JNJ': (0.00307, 0.00314),
    'JPM': (0.00559, 0.00565),
    'KO': (0.00281, 0.00272),
    'MSFT': (0.00349, 0.00350),
    'NVDA': (0.01112, 0.01137),
    'PG': (0.00284, 0.00283),
    'SPY': (0.00218, 0.00237),
    'WMT': (0.00401, 0.00408),
    'XOM': (0.00327, 0.00322)
}

# the warning: Part 9's label and model
part9_label = "jump: vol_next more than 1.5 times vol_20d"
part9_auc = 0.8892              # one column, logistic regression, the test block
part9_folds_auc = 0.7923        # the same model on the folds
part9_wide_auc = 0.7458         # twenty columns, C chosen on the folds, the test block
part9_wide_folds_auc = 0.7547   # the same model on the folds
part9_threshold = 0.25          # chosen from the desk's costs: a miss 5, a false alarm 1

print("Loaded prices:", prices.shape[0], "rows")
print("Instruments  :", ", ".join(TICKERS))
print()
print("After Part 9 the risk report holds, on Apple:")
print(f"  a forecast: one column, RMSE {part6_one_rmse:.5f} on the test block and {part6_one_folds:.5f} on the folds")
print(f"              (ridge on twenty columns: {part6_ridge_rmse:.5f} and {part6_ridge_folds:.5f})")
print(f"  a warning : one column, AUC {part9_auc:.3f} on the test block, run at a threshold of {part9_threshold}")
print()
print("Every model in it is linear in its columns. Today the model changes.")

---

### Q1 · Where the report stands

Write `wide_table(ticker)` returning the table Parts 6, 8 and 9 all worked on, for one instrument: the six volatility windows `[5, 10, 20, 40, 60, 120]` as `vol_<w>d`, the three return windows `[5, 20, 60]` as `ret_<w>d`, `up_20d`, the 20-day volatility of every **other** instrument as `<ticker>_vol`, the target `vol_next`, and incomplete rows dropped. Build Apple's table with it, store the twenty feature names in `columns`, and split at the end of 2022. Then refit the report's forecast, the one-column linear regression, as `one_model`, and check its test RMSE, `one_rmse`, against `part6_one_rmse`.

In [ ]:
def wide_table(ticker):
    """..."""
    ...


table = ...
columns = ...
train = ...
test = ...

one_model = ...
...
one_rmse = ...

print(one_rmse)
print('matches the report:', ...)

<details>
<summary>💡 Hint 1</summary>

Column names are text built from the number: `'vol_' + str(w) + 'd'`. Inside the last loop, `if t != ticker:`. The target is `rets[ticker].rolling(20).std().shift(-20)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`columns = list(table.columns[:-1])`, and the check is `abs(one_rmse - part6_one_rmse) < 0.00001`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def wide_table(ticker):
    """The report's twenty columns and the target for one instrument, in plain decimals."""
    frame = pd.DataFrame()
    for w in [5, 10, 20, 40, 60, 120]:
        frame['vol_' + str(w) + 'd'] = rets[ticker].rolling(w).std()
    for w in [5, 20, 60]:
        frame['ret_' + str(w) + 'd'] = rets[ticker].rolling(w).mean()
    frame['up_20d'] = (rets[ticker] > 0).rolling(20).mean()
    for t in TICKERS:
        if t != ticker:
            frame[t + '_vol'] = rets[t].rolling(20).std()
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    return frame.dropna()


table = wide_table('AAPL')
columns = list(table.columns[:-1])
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]

one_model = LinearRegression()
one_model.fit(train[['vol_20d']], train['vol_next'])
one_rmse = rmse(test['vol_next'], one_model.predict(test[['vol_20d']]))

print(one_rmse)
print('matches the report:', abs(one_rmse - part6_one_rmse) < 0.00001)
```

0.00412, and `True`: 2,376 rows, 1,894 to fit on and 482 to check on. The table is a function now because Q11 runs it for every instrument; nothing else about it has changed. The forecast has two scores in the report, 0.00412 on the test block, which is two calm years, and 0.00755 on the folds, which include 2020. Every model in this part is held to both, and one that wins on one and loses on the other has not won.

</details>

---

### Q2 · One question, checked by hand

Fit a tree with one question, `DecisionTreeRegressor(max_depth=1, random_state=0)`, on `vol_20d` alone, as `stump`. Its cut is `stump.tree_.threshold[0]`. With a mask on the training days, count the days on each side of the cut and print the mean `vol_next` on each side: those are the stump's two forecasts. Finish with its test RMSE beside `one_rmse`.

In [ ]:
stump = ...
...
cut = ...
calm = ...

print('cut:', cut)
print('calm side:', ..., 'days, forecast', ...)
print('busy side:', ..., 'days, forecast', ...)
stump_rmse = ...
print('stump:', stump_rmse, ' one column:', one_rmse)

<details>
<summary>💡 Hint 1</summary>

`calm = train['vol_20d'] <= cut` is True on the calm side. `calm.sum()` counts it and `~calm` is the other side.

</details>

<details>
<summary>💡 Hint 2</summary>

`train.loc[calm, 'vol_next'].mean()` is the calm forecast.

</details>

<details>
<summary>✅ Solution</summary>

```python
stump = DecisionTreeRegressor(max_depth=1, random_state=0)
stump.fit(train[['vol_20d']], train['vol_next'])
cut = stump.tree_.threshold[0]
calm = train['vol_20d'] <= cut

print('cut:', round(cut, 5))
print('calm side:', calm.sum(), 'days, forecast', round(train.loc[calm, 'vol_next'].mean(), 5))
print('busy side:', (~calm).sum(), 'days, forecast', round(train.loc[~calm, 'vol_next'].mean(), 5))
stump_rmse = rmse(test['vol_next'], stump.predict(test[['vol_20d']]))
print('stump:', stump_rmse, ' one column:', one_rmse)
```

A cut at 0.0175: 1,181 calm days forecast 0.01437 and 713 busy days forecast 0.02204, the means of their boxes. The stump scores 0.00416 against 0.00412. Two numbers come within 0.00004 of a fitted slope, partly because only 48 of the 482 test days fell on the busy side.

</details>

---

### Q3 · Every column for one question

Now offer the stump all twenty columns, as `stump20`. Print which column it asks about, `columns[stump20.tree_.feature[0]]`, its cut, how many test days land on the busy side of that cut, and its test RMSE beside the RMSE of forecasting the training average for every test day.

In [ ]:
stump20 = ...
...
asked = ...

print('asks about:', asked)
print('cut:', ...)
print('test days on the busy side:', ...)
print('stump20:', ..., ' average:', ...)

<details>
<summary>💡 Hint 1</summary>

The busy side is `test[asked] > stump20.tree_.threshold[0]`; `.sum()` counts it.

</details>

<details>
<summary>💡 Hint 2</summary>

The average forecast for every test day is `np.full(len(test), train['vol_next'].mean())`.

</details>

<details>
<summary>✅ Solution</summary>

```python
stump20 = DecisionTreeRegressor(max_depth=1, random_state=0)
stump20.fit(train[columns], train['vol_next'])
asked = columns[stump20.tree_.feature[0]]

print('asks about:', asked)
print('cut:', round(stump20.tree_.threshold[0], 5))
print('test days on the busy side:', (test[asked] > stump20.tree_.threshold[0]).sum())
print('stump20:', rmse(test['vol_next'], stump20.predict(test[columns])),
      ' average:', rmse(test['vol_next'], np.full(len(test), train['vol_next'].mean())))
```

`DIS_vol` at 0.01256: Apple's first question is about Disney. On the training days that cut left an RMSE of 0.00741, a little below the 0.00745 of Apple's own volatility, so the tree took it. On the test days 330 of 482 land on the busy side, and the stump scores 0.00750, worse than the average at 0.00533. The best question on eight years was not the best question on the next two.

</details>

---

### Q4 · Grow it until it fits

For depths 1, 2, 3, 4, 6, 10 and no limit (`None`), fit a tree on all twenty columns and collect the number of leaves, the training RMSE and the test RMSE in a DataFrame `growth`, one row per depth.

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 6, 10, None]:
    ...

growth = ...
print(growth)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, fit the tree and append a dictionary: `{'depth': str(depth), 'leaves': t.get_n_leaves(), 'train': ..., 'test': ...}`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.DataFrame(rows).set_index('depth')` turns the list of dictionaries into the table.

</details>

<details>
<summary>✅ Solution</summary>

```python
rows = []
for depth in [1, 2, 3, 4, 6, 10, None]:
    t = DecisionTreeRegressor(max_depth=depth, random_state=0)
    t.fit(train[columns], train['vol_next'])
    rows.append({'depth': str(depth), 'leaves': t.get_n_leaves(),
                 'train': rmse(train['vol_next'], t.predict(train[columns])),
                 'test': rmse(test['vol_next'], t.predict(test[columns]))})

growth = pd.DataFrame(rows).set_index('depth')
print(growth.round(5))
```

With no limit the training error is zero to nine decimals: 1,891 leaves for 1,894 days, at depth 27. The test error is lowest at depth 3, 0.00558, and 0.00975 with no limit. Every tree in the table is worse than forecasting the training average (0.00533). On twenty columns and eight years, one tree has too many ways to be wrong.

</details>

---

### Q5 · Let the folds choose the size

Search `max_depth` over `[1, 2, 3, 4, 6, 8]` as `depth_search`, and separately `min_samples_leaf` over `[20, 50, 100, 200, 400]` as `leaf_search`, each on `folds`. For each search print the best setting, its mean fold RMSE and its test RMSE, and compare the fold scores with `part6_one_folds`.

In [ ]:
depth_search = ...
...
leaf_search = ...
...

print(...)
print(...)
print('one column on the folds:', part6_one_folds)

<details>
<summary>💡 Hint 1</summary>

`GridSearchCV(DecisionTreeRegressor(random_state=0), {'max_depth': [...]}, cv=folds, scoring='neg_root_mean_squared_error')`, fitted on the twenty columns.

</details>

<details>
<summary>💡 Hint 2</summary>

A loop `for s in [depth_search, leaf_search]:` prints both with one line.

</details>

<details>
<summary>✅ Solution</summary>

```python
depth_search = GridSearchCV(DecisionTreeRegressor(random_state=0), {'max_depth': [1, 2, 3, 4, 6, 8]},
                            cv=folds, scoring='neg_root_mean_squared_error')
depth_search.fit(train[columns], train['vol_next'])
leaf_search = GridSearchCV(DecisionTreeRegressor(random_state=0),
                           {'min_samples_leaf': [20, 50, 100, 200, 400]},
                           cv=folds, scoring='neg_root_mean_squared_error')
leaf_search.fit(train[columns], train['vol_next'])

for s in [depth_search, leaf_search]:
    print(s.best_params_, round(-s.best_score_, 5), round(rmse(test['vol_next'], s.predict(test[columns])), 5))
print('one column on the folds:', part6_one_folds)
```

The folds choose depth 1, which is the Disney stump of Q3, at 0.00829, and leaves of at least 200 days at 0.00836: a tree of 7 leaves that scores 0.00565 on the test block. Both are behind the one column on the folds (0.00755). Choosing the size honestly limits the damage; it does not find a tree that helps.

</details>

---

### Q6 · Two columns, four leaves

Go back to the two columns the lecture's tree used. Fit a tree of depth 2 on `pair` as `pair_tree` and draw it with `plot_tree`, without the error line and with four decimals. List the column behind each of its three questions, then compare it with the one-column model on the test block and on the folds.

In [ ]:
pair = ['vol_20d', 'ret_20d']
pair_tree = ...
...

plt.figure(figsize=(11, 3.6))
...
plt.show()

pair_asked = ...
pair_rmse = ...
pair_folds = ...

print('questions about:', pair_asked)
print('test :', pair_rmse, ' one column:', one_rmse)
print('folds:', pair_folds, ' one column:', part6_one_folds)

<details>
<summary>💡 Hint 1</summary>

`plot_tree(pair_tree, feature_names=pair, filled=True, impurity=False, precision=4, fontsize=9)`. `pair_tree.tree_.feature` holds a column number for every box, and -2 for a leaf, so `[pair[i] for i in pair_tree.tree_.feature if i >= 0]` keeps the questions.

</details>

<details>
<summary>💡 Hint 2</summary>

Score it on the folds with `cross_val_score` on a fresh `DecisionTreeRegressor(max_depth=2, random_state=0)`, negated and averaged.

</details>

<details>
<summary>✅ Solution</summary>

```python
pair = ['vol_20d', 'ret_20d']
pair_tree = DecisionTreeRegressor(max_depth=2, random_state=0)
pair_tree.fit(train[pair], train['vol_next'])

plt.figure(figsize=(11, 3.6))
plot_tree(pair_tree, feature_names=pair, filled=True, impurity=False, precision=4, fontsize=9)
plt.show()

pair_asked = [pair[i] for i in pair_tree.tree_.feature if i >= 0]
pair_rmse = rmse(test['vol_next'], pair_tree.predict(test[pair]))
pair_folds = -cross_val_score(DecisionTreeRegressor(max_depth=2, random_state=0), train[pair],
                              train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error').mean()

print('questions about:', pair_asked)
print('test :', pair_rmse, ' one column:', one_rmse)
print('folds:', pair_folds, ' one column:', part6_one_folds)
```

All three questions are about `vol_20d`. The root asks the stump's question again, at 0.0175; the calm side then cuts at 0.0147 and the busy side at 0.0344, a level only 39 training days reached. On Apple the tree never asks about the direction: it is a staircase of four forecasts on one column, rising from 0.0135 to 0.0336. It beats the one-column model on the test block by 0.00001 and loses on the folds by 0.0006. A win that small on two calm years, against a loss on the folds, is not a reason to change the report's model.

</details>

---

### Q7 · Bagging, then a forest

Fit two forests of 100 trees on the twenty columns with `random_state=0`: `bagged` with `max_features=1.0`, where every question sees every column, which is plain bagging, and `forest` with `max_features='sqrt'`. Time each fit with `time.perf_counter()`, and print the seconds and the test RMSE of each.

In [ ]:
start = ...
bagged = ...
...
bagged_seconds = ...

start = ...
forest = ...
...
forest_seconds = ...

print('bagging:', ..., ...)
print('forest :', ..., ...)

<details>
<summary>💡 Hint</summary>

`start = time.perf_counter()` before the fit and `time.perf_counter() - start` after it.

</details>

<details>
<summary>✅ Solution</summary>

```python
start = time.perf_counter()
bagged = RandomForestRegressor(n_estimators=100, max_features=1.0, random_state=0)
bagged.fit(train[columns], train['vol_next'])
bagged_seconds = time.perf_counter() - start

start = time.perf_counter()
forest = RandomForestRegressor(n_estimators=100, max_features='sqrt', random_state=0)
forest.fit(train[columns], train['vol_next'])
forest_seconds = time.perf_counter() - start

print('bagging:', round(bagged_seconds, 2), 's', round(rmse(test['vol_next'], bagged.predict(test[columns])), 5))
print('forest :', round(forest_seconds, 2), 's', round(rmse(test['vol_next'], forest.predict(test[columns])), 5))
```

Bagging scores 0.00675 and the forest 0.00528. The forest was about 4 times quicker to fit, because each question looks at 4 columns instead of 20; the seconds depend on your machine, the ratio much less. The forest is the first model built from trees in this part to beat the training average (0.00533) on twenty columns, and it is still well behind the one column.

</details>

---

### Q8 · Out-of-bag, or the folds

Refit the forest from Q7 with `oob_score=True`, and print its out-of-bag RMSE on the training days beside its mean RMSE on the folds, stored as `forest_folds`. Say in a comment which of the two belongs in the report.

In [ ]:
forest = ...
...
oob_rmse = ...
forest_folds = ...

print('out-of-bag:', oob_rmse)
print('folds     :', forest_folds, ' one column:', part6_one_folds)
# which one belongs in the report, and why?

<details>
<summary>💡 Hint 1</summary>

`forest.oob_prediction_` holds a forecast for every training day from the trees that did not see it.

</details>

<details>
<summary>💡 Hint 2</summary>

Score a fresh forest with the same settings on the folds; `oob_score` is not needed there.

</details>

<details>
<summary>✅ Solution</summary>

```python
forest = RandomForestRegressor(n_estimators=100, max_features='sqrt', oob_score=True, random_state=0)
forest.fit(train[columns], train['vol_next'])
oob_rmse = rmse(train['vol_next'], forest.oob_prediction_)
forest_folds = -cross_val_score(RandomForestRegressor(n_estimators=100, max_features='sqrt',
                                                      random_state=0),
                                train[columns], train['vol_next'], cv=folds,
                                scoring='neg_root_mean_squared_error').mean()

print('out-of-bag:', oob_rmse)
print('folds     :', forest_folds, ' one column:', part6_one_folds)
# The folds. Out-of-bag scores each day with trees that saw the days either
# side of it, and neighbouring days share most of a 20-day window.
```

0.00267 out-of-bag against 0.00825 on the folds. Out-of-bag makes the forest look 3 times better than it is, and better than anything else in the report; on the folds it is behind the one column (0.00755). A score the model computes on itself is honest only when the rows are independent, and days are not.

</details>

---

### Q9 · What a grid will cost, before you run it

The grid below has two settings. Count its combinations and the fits it needs: five folds for each combination, plus one refit at the end. Then time one forest of 100 trees at the slowest corner of the grid, 20 columns and leaves of 5, and at the quickest, 1 column and leaves of 200, and turn the two into a range for the whole search on one core.

In [ ]:
grid = {'max_features': [1, 3, 5, 10, 20], 'min_samples_leaf': [5, 20, 50, 100, 200]}

combinations = ...
fits = ...

timings = {}
for setting in [(20, 5), (1, 200)]:
    ...

print(combinations, 'combinations,', fits, 'fits')
print(timings)
print('between', ..., 'and', ..., 'seconds on one core')

<details>
<summary>💡 Hint 1</summary>

`combinations = len(grid['max_features']) * len(grid['min_samples_leaf'])` and `fits = combinations * 5 + 1`.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the loop, build the forest with `max_features=setting[0]` and `min_samples_leaf=setting[1]`, time its fit, and store the seconds in `timings[setting]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
grid = {'max_features': [1, 3, 5, 10, 20], 'min_samples_leaf': [5, 20, 50, 100, 200]}

combinations = len(grid['max_features']) * len(grid['min_samples_leaf'])
fits = combinations * 5 + 1

timings = {}
for setting in [(20, 5), (1, 200)]:
    f = RandomForestRegressor(n_estimators=100, max_features=setting[0],
                              min_samples_leaf=setting[1], random_state=0)
    start = time.perf_counter()
    f.fit(train[columns], train['vol_next'])
    timings[setting] = round(time.perf_counter() - start, 2)

print(combinations, 'combinations,', fits, 'fits')
print(timings)
print('between', round(fits * timings[(1, 200)]), 'and', round(fits * timings[(20, 5)]),
      'seconds on one core')
```

25 combinations and 126 fits, which is 12,600 trees. The slow corner took 1.81 seconds here and the quick one 0.12, so the search lies somewhere between 15 and 229 seconds on one core. The fits on the folds use fewer rows than the full training block, so the real figure sits lower. A range is the honest answer, and it is enough to decide whether to wait.

</details>

---

### Q10 · Run it, on every core

Run the search as `forest_search`, with `GridSearchCV` on `folds` and the forest set to `n_jobs=-1` so that its trees grow on every core, and time it. Print the best settings, the mean fold RMSE, the test RMSE as `forest_rmse`, and the seconds, with the report's two forecasts beside the scores. It takes about a minute on Colab.

In [ ]:
start = ...
forest_search = ...
...
search_seconds = ...
forest_rmse = ...

print(...)
print('folds:', ..., ' ridge:', part6_ridge_folds, ' one column:', part6_one_folds)
print('test :', forest_rmse, ' ridge:', part6_ridge_rmse, ' one column:', one_rmse)
print('time :', search_seconds)

<details>
<summary>💡 Hint 1</summary>

`GridSearchCV(RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1), grid, cv=folds, scoring='neg_root_mean_squared_error')`.

</details>

<details>
<summary>💡 Hint 2</summary>

The refitted best forest forecasts with `forest_search.predict(...)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
start = time.perf_counter()
forest_search = GridSearchCV(RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1),
                             grid, cv=folds, scoring='neg_root_mean_squared_error')
forest_search.fit(train[columns], train['vol_next'])
search_seconds = time.perf_counter() - start
forest_rmse = rmse(test['vol_next'], forest_search.predict(test[columns]))

print(forest_search.best_params_)
print('folds:', -forest_search.best_score_, ' ridge:', part6_ridge_folds, ' one column:', part6_one_folds)
print('test :', forest_rmse, ' ridge:', part6_ridge_rmse, ' one column:', one_rmse)
print('time :', search_seconds)
```

Five columns per question and leaves of at least 100 days. The tuned forest scores 0.00757 on the folds against ridge's 0.00761, and 0.00462 on the test block against ridge's 0.00460: level with ridge on both yardsticks. The settings alone took the forest from bagging's 0.00675 to 0.00462, almost a third less error. The one column still leads on both, at 0.00755 and 0.00412. The search took 18 seconds on a machine with many cores.

</details>

---

### Q11 · The same forest across the desk

Using `wide_table` from Q1, fit a forest with the settings the search chose on Apple for every instrument, and store its test RMSE in `desk_forest`. `part6_desk` holds each instrument's ridge and one-column RMSE, as the report established them. Count the instruments where the forest beats each.

In [ ]:
desk_forest = {}
for ticker in TICKERS:
    ...

beats_ridge = ...
beats_one = ...

print('forest beats ridge on', beats_ridge, 'of', len(TICKERS))
print('forest beats one column on', beats_one, 'of', len(TICKERS))

<details>
<summary>💡 Hint 1</summary>

Inside the loop: `frame = wide_table(ticker)`, split it with `.loc`, `cols = list(frame.columns[:-1])`, and build the forest with `max_features=forest_search.best_params_['max_features']` and the same for `min_samples_leaf`.

</details>

<details>
<summary>💡 Hint 2</summary>

`sum(1 for t in TICKERS if desk_forest[t] < part6_desk[t][0])` counts the wins over ridge; `[1]` is the one column.

</details>

<details>
<summary>✅ Solution</summary>

```python
desk_forest = {}
for ticker in TICKERS:
    frame = wide_table(ticker)
    tr, te = frame.loc[:'2022-12-31'], frame.loc['2023-01-01':]
    cols = list(frame.columns[:-1])
    f = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1,
                              max_features=forest_search.best_params_['max_features'],
                              min_samples_leaf=forest_search.best_params_['min_samples_leaf'])
    f.fit(tr[cols], tr['vol_next'])
    desk_forest[ticker] = rmse(te['vol_next'], f.predict(te[cols]))

beats_ridge = sum(1 for t in TICKERS if desk_forest[t] < part6_desk[t][0])
beats_one = sum(1 for t in TICKERS if desk_forest[t] < part6_desk[t][1])

print('forest beats ridge on', beats_ridge, 'of', len(TICKERS))
print('forest beats one column on', beats_one, 'of', len(TICKERS))
```

The forest beats ridge on 5 of 11, DIS, MSFT, NVDA, WMT and XOM, and the one column on 4. The settings were chosen on Apple and used everywhere, and they may not suit the other ten. Level with ridge on Apple and better on about half the desk: that is the forest's honest result on this target, not the win the lecture's forest had on the credit table, where the rows were separate borrowers.

</details>

---

### Q12 · Draw the forecasts

One figure of the test years: what happened as a black line, and the one-column, stump and tuned forest forecasts as three more lines, with a legend and a title. Draw the stump with `drawstyle='steps-post'` so its two levels show as steps.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
...
...
...
...
ax.set_ylabel('20-day volatility')
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(test.index, test['vol_next'], color='black', label='what happened')`, then each forecast with its own `label`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.legend()` and `ax.set_title(..., loc='left')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(test.index, test['vol_next'], color='black', linewidth=1.2, label='what happened')
ax.plot(test.index, one_model.predict(test[['vol_20d']]), label='one column')
ax.plot(test.index, stump.predict(test[['vol_20d']]), drawstyle='steps-post', label='stump')
ax.plot(test.index, forest_search.predict(test[columns]), label='tuned forest')
ax.legend()
ax.set_ylabel('20-day volatility')
ax.set_title('Apple, 2023 to 2024: three forecasts against what happened', loc='left')
plt.show()
```

All three forecasts sit above what happened on about three test days in four, because the test years were calmer than the training years: on average the one-column model forecast 0.0155 and the forest 0.0156, where 0.0131 happened. The stump holds one level on all but 48 days. The forest moves more than the line and does not follow the turns any better.

</details>

---

### Q13 · The same forest, once, on the warning

The report's other half is Part 9's jump warning. Make the jump label for the training and test days as `train_jump` and `test_jump`, and fit a `RandomForestClassifier` of 100 trees with the settings `forest_search` chose for the number, as `jump_forest`. Print its test AUC, `forest_auc`, and its mean fold AUC, `forest_folds_auc`, each beside Part 9's two models.

In [ ]:
train_jump = ...
test_jump = ...

jump_forest = ...
...
forest_auc = ...
forest_folds_auc = ...

print('test :', forest_auc, ' Part 9, one column:', part9_auc, ' twenty:', part9_wide_auc)
print('folds:', forest_folds_auc, ' Part 9, one column:', part9_folds_auc, ' twenty:', part9_wide_folds_auc)

<details>
<summary>💡 Hint 1</summary>

The label is `(train['vol_next'] > 1.5 * train['vol_20d']).astype(int)`, and the same on `test`.

</details>

<details>
<summary>💡 Hint 2</summary>

`roc_auc_score(test_jump, jump_forest.predict_proba(test[columns])[:, 1])`; on the folds, `cross_val_score(..., scoring='roc_auc')` with no minus sign, because a larger AUC is better.

</details>

<details>
<summary>✅ Solution</summary>

```python
train_jump = (train['vol_next'] > 1.5 * train['vol_20d']).astype(int)
test_jump = (test['vol_next'] > 1.5 * test['vol_20d']).astype(int)

jump_forest = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1,
                                     max_features=forest_search.best_params_['max_features'],
                                     min_samples_leaf=forest_search.best_params_['min_samples_leaf'])
jump_forest.fit(train[columns], train_jump)
forest_auc = roc_auc_score(test_jump, jump_forest.predict_proba(test[columns])[:, 1])
forest_folds_auc = cross_val_score(jump_forest, train[columns], train_jump, cv=folds,
                                   scoring='roc_auc').mean()

print('test :', forest_auc, ' Part 9, one column:', part9_auc, ' twenty:', part9_wide_auc)
print('folds:', forest_folds_auc, ' Part 9, one column:', part9_folds_auc, ' twenty:', part9_wide_folds_auc)
```

0.869 on the test block, between Part 9's one column (0.889) and its twenty (0.746). On the folds 0.682, below both (0.792 and 0.755). The test block made the forest look like a better use of the twenty columns than Part 9's wide model; the folds do not agree, and they cover more years. The warning stays as Part 9 left it, threshold and all.

</details>

---

### Q14 · Write down what you would defend

Collect what Part 11 established into a dictionary `report`, and write `summarise(report)`: it prints one line per entry and ends with two verdicts, one on the volatility forecast and one on the jump warning.

In [ ]:
report = {
    'target': ...,
    'one_column_rmse': ...,
    'ridge_rmse': ...,
    'stump_rmse': ...,
    'best_single_tree_rmse': ...,
    'forest_untuned_rmse': ...,
    'forest_tuned_rmse': ...,
    'forest_settings': ...,
    'forest_beats_ridge_on': ...,
    'warning_logistic_auc': ...,
    'warning_forest_auc': ...,
}


def summarise(r):
    """..."""
    ...


summarise(report)

<details>
<summary>💡 Hint 1</summary>

Every value is a number or a short string you already have: `part5_target`, `one_rmse`, `part6_ridge_rmse`, `stump_rmse`, the smallest entry of `growth['test']`, the forest from Q8, `forest_rmse`, `forest_search.best_params_`, `beats_ridge`, `part9_auc` and `forest_auc`.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the function, `for key, value in r.items():` and a `print` with an f-string, then two `if` statements for the verdicts.

</details>

<details>
<summary>✅ Solution</summary>

```python
report = {
    'target': part5_target,
    'one_column_rmse': round(one_rmse, 5),
    'ridge_rmse': part6_ridge_rmse,
    'stump_rmse': round(stump_rmse, 5),
    'best_single_tree_rmse': round(growth['test'].min(), 5),
    'forest_untuned_rmse': round(rmse(test['vol_next'], forest.predict(test[columns])), 5),
    'forest_tuned_rmse': round(forest_rmse, 5),
    'forest_settings': forest_search.best_params_,
    'forest_beats_ridge_on': f'{beats_ridge} of {len(TICKERS)} instruments',
    'warning_logistic_auc': part9_auc,
    'warning_forest_auc': round(forest_auc, 4),
}


def summarise(r):
    """Print the report and the two verdicts it supports."""
    for key, value in r.items():
        print(f"{key:<24}{value}")
    print()
    if r['forest_tuned_rmse'] < r['one_column_rmse']:
        print('Forecast: the tuned forest replaces the one-column model.')
    else:
        print('Forecast: the one-column model stays.')
    if r['warning_forest_auc'] > r['warning_logistic_auc']:
        print('Warning: the forest replaces the one-column logistic regression.')
    else:
        print('Warning: the one-column logistic regression of Part 9 stays.')


summarise(report)
```

The report keeps both of its models, and now it can say why with a flexible model behind each verdict. The trees found patterns in the training years, Disney's volatility among them, that did not carry into the next two; the forest, tuned on the folds, drew level with ridge and no further. The untuned forest is in the report on purpose: next to the tuned one it shows how much of a forest's score comes from its settings.

</details>

---

## 📦 What you now have

| what | where |
|:--|:--|
| the report's table as a function, Apple's split and its forecast | `wide_table`, `table`, `columns`, `train`, `test`, `one_model`, `one_rmse` |
| one question, on one column and on twenty | `stump`, `cut`, `stump_rmse`, `stump20` |
| trees grown deeper, and sized on the folds | `growth`, `depth_search`, `leaf_search` |
| the staircase on two columns, drawn | `pair_tree`, `pair_rmse`, `pair_folds` |
| bagging and a forest, timed | `bagged`, `forest`, `bagged_seconds`, `forest_seconds` |
| out-of-bag against the folds | `oob_rmse`, `forest_folds` |
| the grid, costed and searched | `grid`, `timings`, `forest_search`, `forest_rmse`, `search_seconds` |
| the whole desk | `desk_forest`, `beats_ridge`, `beats_one` |
| the same forest on the jump label | `train_jump`, `test_jump`, `jump_forest`, `forest_auc`, `forest_folds_auc` |
| the thing you would defend | `report` |

## What changed since Part 9

- **A new kind of model.** Everything in the report was linear in its columns. A tree asks questions instead, and a forest averages many trees; both were held to the numbers the report had already established, on the test block and on the folds.
- **More columns made a single tree worse.** Offered twenty columns, a tree asks about Disney first (Q3) and is worse than the training average (0.00533) at every depth (Q4). A stump on `vol_20d` alone scores 0.00416 with two numbers, against 0.00412 for the line (Q2).
- **A forest tuned on the folds draws level with ridge.** 0.00462 against 0.00460 on the test block and 0.00757 against 0.00761 on the folds (Q10), and better than ridge on 5 of the 11 instruments (Q11). The search was costed before it ran (Q9).
- **Out-of-bag flattered the forest.** 0.00267 against 0.00825 on the folds, because days are not independent rows (Q8).
- **The warning is unchanged.** The same forest scores 0.869 on Part 9's jump label and 0.682 on the folds, below the one column on both (Q13).

## Where this leaves the risk report

Nothing in the report changes, and that is the finding. The volatility forecast is still the one-column model of Part 5, and the jump warning is still Part 9's one-column logistic regression at a threshold of 0.25. What changed is how much stands behind them. The twenty columns have now been offered to a model that can ask different questions of calm and busy weeks, and on Apple they still do not beat this month's volatility. The desk has a second family of models it can fit, tune, time and cost, and on about half of its instruments the forest is the better of the two flexible forecasts.

**Next part:** boosting, where each tree is grown to correct the ones before it, and a look at which columns the models lean on.